In [ ]:
# Parameters cell — Papermill overrides these at runtime
dataset_size = 10
start_base = 2440000
size_increment = 1000
sample_set = "1354-VO-KE-DONNELLY-VMF00281"
output_csv = "memory_benchmarks.csv"

In [1]:
import malariagen_data
from pathlib import Path

af1 = malariagen_data.Af1()

def run_snp_calls(sample_id, path_distinguisher, region):
    local_vcf_path = Path("local_igv_vcfs", f"{sample_id}_{path_distinguisher}.vcf.gz")
    #local_vcf_path.parent.mkdir(parents=True, exist_ok=True),
    af1.snp_calls_to_vcf(
        output_path=str(local_vcf_path),
        region=region,
        sample_sets=[sample_set],
        sample_query=f"sample_id == '{sample_id}'",
        overwrite=True,
    )
        
def run_snp_calls_batched(start_base, size_increment, dataset_size, sample_set):
    sample_id = str(
        af1.sample_metadata(sample_sets=sample_set).iloc[0]["sample_id"]
    )
    region_indices = [start_base + i*size_increment for i in range(dataset_size)]
    regions = [f"2RL:{i}-{i+size_increment}" for i in region_indices]
    local_vcf_path_distinguishers = [str(i*size_increment) for i in range(dataset_size)]
    for path_distinguisher, region in zip(local_vcf_path_distinguishers, regions):
        local_vcf_path = Path("local_igv_vcfs", f"{sample_id}_{path_distinguisher}.vcf.gz")
        local_vcf_path.parent.mkdir(parents=True, exist_ok=True),
        run_snp_calls(sample_id, path_distinguisher, region)


def view_alignments(sample_id, output_path, region):
    af1.view_alignments(
        sample=sample_id,
        region=region,
        snp_vcf_path=output_path,
    )

def view_alignments_batched(start_base, size_increment, dataset_size, sample_set):
    sample_id = str(
        af1.sample_metadata(sample_sets=sample_set).iloc[0]["sample_id"]
    )
    region_indices = [start_base + i*size_increment for i in range(dataset_size)]
    regions = [f"2RL:{i}-{i+size_increment}" for i in region_indices]
    local_vcf_path_distinguishers = [str(i*size_increment) for i in range(dataset_size)]
    for path_distinguisher, region in zip(local_vcf_path_distinguishers, regions):
        local_vcf_path = Path("local_igv_vcfs", f"{sample_id}_{path_distinguisher}.vcf.gz")
        local_vcf_path.parent.mkdir(parents=True, exist_ok=True),
        output_path=str(local_vcf_path)
        print(f"Memory usage for region {region}")
        view_alignments(sample_id, output_path, region)

In [ ]:
import os
import csv
import filelock  # pip install filelock (prevents race conditions if running concurrently)
from memory_profiler import memory_usage
import time


# memory_usage returns peak RAM in MiB during execution
start_time = time.perf_counter()
peak_memory_mib = memory_usage(
    (run_snp_calls_batched, (start_base, size_increment, dataset_size, sample_set)),
    max_usage=True,
)
wall_time_sec = time.perf_counter() - start_time

# Record dictionary
memory_records = {
    "dataset_size": dataset_size,
    "size_increment": size_increment,
    "start_base": start_base,
    "sample_set": sample_set,
    "peak_memory_mib": round(peak_memory_mib, 2),
    "runtime_sec": round(wall_time_sec, 4),
}

# memory_usage returns peak RAM in MiB during execution
start_time = time.perf_counter()
peak_memory_mib = memory_usage(
    (view_alignments_batched, (start_base, size_increment, dataset_size, sample_set)),
    max_usage=True,
)
wall_time_sec = time.perf_counter() - start_time

# Record dictionary
igv_view_records = {
    "dataset_size": dataset_size,
    "size_increment": size_increment,
    "start_base": start_base,
    "sample_set": sample_set,
    "peak_memory_mib": round(peak_memory_mib, 2),
    "runtime_sec": round(wall_time_sec, 4),
}

# Append safely using a file lock
lock = filelock.FileLock(f"{output_csv}_snp_calls.lock")
with lock:
    file_exists = os.path.isfile(output_csv)
    with open(output_csv, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=memory_records.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(record)

lock = filelock.FileLock(f"{output_csv}_igv_views.lock")
with lock:
    file_exists = os.path.isfile(output_csv)
    with open(output_csv, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=igv_view_records.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(record)

print(f"Recorded: {record}")

In [2]:
#%%capture mem_result

#def test_function():
#    for i in range(10):
#        print(f"Hello {i}")

#%memit test_function()
#%memit test_function()
#%memit test_function()


In [3]:
#with open("mem_profiles.log", "a") as f:
#    f.write(f"Task 1: {mem_result.stdout.strip()}\n")